# 🚢 Titanic — Modelo 3: Voting Classifier (Ensemble)

Este notebook combina os três melhores modelos em um ensemble por votação:

| Modelo | Parâmetros finais |
|---|---|
| Logistic Regression | C=1, penalty='l2', solver='liblinear' |
| Random Forest | n_estimators=75, min_samples_leaf=2, min_samples_split=10 |
| KNN | n_neighbors=5, metric='manhattan', weights='uniform' |

São testadas duas estratégias:
- **Hard Voting** → votação majoritária (cada modelo vota 0 ou 1)
- **Soft Voting** → média das probabilidades

---
> **Fluxo:** `01_EDA` → `02_Logistic_Regression` → `03_Random_Forest` → **`04_Voting_Classifier`**

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score

from data_transform import build_preprocessing_pipeline

## 2. Carregamento dos dados

In [ ]:
df_train  = pd.read_csv('dataset/titanicc/train.csv')
df_test   = pd.read_csv('dataset/titanicc/test.csv')
df_labels = pd.read_csv('dataset/titanicc/gender_submission.csv')

FEATURES = ['PassengerId', 'Pclass', 'Name', 'Sex', 'Age',
            'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

X_train = df_train[FEATURES]
y_train = df_train['Survived']

X_test  = df_test
y_test  = df_labels['Survived'].to_numpy()

## 3. Definição dos Estimadores Base

In [ ]:
# Melhores parâmetros encontrados via GridSearchCV nos notebooks anteriores

logit = LogisticRegression(
    C=1, max_iter=1000, solver='liblinear', penalty='l2'
)

rf = RandomForestClassifier(
    bootstrap=True, max_depth=None, max_features='sqrt',
    random_state=42, min_samples_leaf=2,
    min_samples_split=10, n_estimators=75
)

knn = KNeighborsClassifier(
    metric='manhattan', n_neighbors=5, weights='uniform'
)

estimators = [('lr', logit), ('rf', rf), ('knn', knn)]

## 4. Hard Voting

In [ ]:
voting_hard = VotingClassifier(estimators=estimators, voting='hard')
model_hard  = make_pipeline(build_preprocessing_pipeline(), voting_hard)
model_hard.fit(X_train, y_train)

y_pred_hard = model_hard.predict(X_test)

print(f"Acurácia Hard Voting: {accuracy_score(y_test, y_pred_hard):.4f}")
print()
print(classification_report(y_test, y_pred_hard, target_names=['Morreu', 'Sobreviveu']))

## 5. Soft Voting

In [ ]:
voting_soft = VotingClassifier(estimators=estimators, voting='soft')
model_soft  = make_pipeline(build_preprocessing_pipeline(), voting_soft)
model_soft.fit(X_train, y_train)

y_pred_soft = model_soft.predict(X_test)

print(f"Acurácia Soft Voting: {accuracy_score(y_test, y_pred_soft):.4f}")
print()
print(classification_report(y_test, y_pred_soft, target_names=['Morreu', 'Sobreviveu']))

## 6. Comparação Visual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_hard,
    display_labels=['Morreu', 'Sobreviveu'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'Hard Voting  (acc={accuracy_score(y_test, y_pred_hard):.3f})', fontsize=12)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_soft,
    display_labels=['Morreu', 'Sobreviveu'],
    cmap='Greens', ax=axes[1]
)
axes[1].set_title(f'Soft Voting  (acc={accuracy_score(y_test, y_pred_soft):.3f})', fontsize=12)

plt.suptitle('Voting Classifier — Comparação Hard vs Soft', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Resumo Comparativo dos Modelos

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

resultados = {
    'Soft Voting': y_pred_soft,
    'Hard Voting': y_pred_hard,
}

rows = []
for nome, y_pred in resultados.items():
    rows.append({
        'Modelo'   : nome,
        'Acurácia' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
        'F1-Score' : round(f1_score(y_test, y_pred), 4),
    })

df_results = pd.DataFrame(rows).set_index('Modelo')
print(df_results.to_string())

---
## ✅ Conclusão Final

O **Voting Classifier** combina os pontos fortes de três algoritmos distintos:
- **Regressão Logística** — modelo linear, boa interpretabilidade
- **Random Forest** — robusto a outliers e não-linearidades
- **KNN** — baseado em similaridade local

O Soft Voting tende a ser mais estável pois considera as probabilidades em vez de votos binários, aproveitando melhor a confiança de cada modelo em suas previsões.